# 1. Text Clustering Pipeline

This notebook runs the UMAP + HDBSCAN clustering pipeline on the EMR text fields to group similar faults, and uses Ollama to label the clusters.

In [1]:
import os
import sys
import pandas as pd

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings
from src.text_clustering import run_clustering_pipeline
from src.transform import load_emr_data

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


In [2]:
print("1. Loading Data...")
file_path = os.path.join(path_prefix, settings.data_dir, settings.emr_file_name)
df = load_emr_data(file_path)
print(f"Loaded {len(df)} records.")

1. Loading Data...
[2026-06-02 14:22:07,247] INFO — src.transform — Loading EMR data: Dashboard EMR(report1776669858353).csv
[2026-06-02 14:22:07,295] INFO — src.transform —   UTF-8 decoding failed — falling back to 'latin1' encoding
[2026-06-02 14:22:07,452] INFO — src.transform —   Loaded 20630 rows, 23 columns.
Loaded 20630 records.


In [3]:
print("2. Running Clustering Pipeline...")
def progress(step, total):
    print(f"Step {step}/{total}")

result = run_clustering_pipeline(df, progress_callback=progress)
print(f"\nClustering completed. Found {result.n_clusters} clusters.")

2. Running Clustering Pipeline...
Step 1/6
Step 2/6


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-06-02 14:22:51,098] INFO — src.text_clustering — Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2 …


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4163.44it/s]


[2026-06-02 14:23:03,154] INFO — src.text_clustering — Embedding 20630 texts …


...

Step 3/6
[2026-06-02 14:29:53,137] INFO — src.text_clustering — UMAP: 384 -> 10 dims ...


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Step 4/6
[2026-06-02 14:30:24,700] INFO — src.text_clustering — HDBSCAN (min_cluster_size=5, min_samples=3) …
[2026-06-02 14:30:30,410] INFO — src.text_clustering — Found 961 clusters, 6514 noise (31.6%)
Step 5/6
[2026-06-02 14:30:30,518] INFO — src.text_clustering — Labeling 1/961 (cluster 0) …
[2026-06-02 14:31:18,139] INFO — src.text_clustering — Cluster 0 -> Preventive Maintenance (1.00)
[2026-06-02 14:31:18,188] INFO — src.text_clustering — Labeling 2/961 (cluster 1) …
[2026-06-02 14:31:37,698] INFO — src.text_clustering — Cluster 1 -> Reinforce Track Frame (1.00)
[2026-06-02 14:31:37,704] INFO — src.text_clustering — Labeling 3/961 (cluster 2) …
[2026-06-02 14:31:43,437] INFO — src.text_clustering — Cluster 2 -> Reinforce Track Frame (1.00)
[2026-06-02 14:31:43,439] INFO — src.text_clustering — Labeling 4/961 (cluster 3) …
[2026-06-02 14:32:04,359] INFO — src.text_clustering — Cluster 3 -> Track Frame Maintenance (1.00)
[2026-06-02 14:32:04,362] INFO — src.text_clustering — Label

d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[2026-06-02 22:40:10,993] INFO — src.text_clustering — Visualization -> D:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\output\cluster_visualization.html

Clustering completed. Found 961 clusters.


In [4]:
print("3. Results Summary...")
print(f"Noise points: {result.noise_count} ({result.noise_pct}%)")
print(f"Interactive visualization saved to: {result.visualization_path}")

print("\nTop clusters:")
for s in result.cluster_summary[:5]:
    print(f"- {s['label']} (Confidence: {s['confidence']:.2f}): {s['size']} records")

3. Results Summary...
Noise points: 6514 (31.58%)
Interactive visualization saved to: D:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\output\cluster_visualization.html

Top clusters:
- Fuel System Clogging (Confidence: 0.95): 186 records
- Engine Noise Abnormal (Confidence: 0.95): 161 records
- Fan Motor Gangguan (Confidence: 0.95): 132 records
- Radiator Leak (Confidence: 1.00): 114 records
- Starting Motor Trouble (Confidence: 0.95): 99 records
